# **2. torch.autograd**
神经网络（NNs）是一系列嵌套函数的集合，这些函数在输入数据上执行。这些函数由参数定义（参数由权重和偏差组成），在 PyTorch 中这些参数存储在张量中。

训练神经网络分为两个步骤：

前向传播：在前向传播中，神经网络对其最佳猜测的正确输出进行判断。它将输入数据通过其每个函数来做出这个猜测。

反向传播：在反向传播中，神经网络根据其猜测中的误差来调整参数。它通过从输出反向遍历，收集误差相对于函数参数的导数（梯度），并使用梯度下降来优化参数。

**示例**
创建一个随机数据张量来表示一个具有 3 个通道、高度和宽度为 64 的单个图像，并为其对应的 label 初始化一些随机值。预训练模型中的标签形状为(1,1000)。

In [17]:
# 引入resnet模型及其推荐权重，设置假想数据与测试集
import torch
from torchvision.models import resnet18, ResNet18_Weights
model = resnet18(weights=ResNet18_Weights.DEFAULT)
data = torch.rand(1,3,64,64)
labels = torch.rand(1, 1000)

In [2]:
# 前向传播
prediction = model(data)

使用模型的预测结果和对应的标签来计算误差（ loss ）。下一步是通过网络反向传播这个误差。当在误差张量上调用 .backward() 时，反向传播就会启动。Autograd 然后计算并存储每个模型参数的梯度，这些梯度存储在参数的 .grad 属性中。

In [3]:
# 损失函数，以及通过反向传播求梯度
loss = (prediction - labels).sum()
loss.backward()

接下来，我们加载一个优化器，这里使用学习率为 0.01、动量为 0.9 的 SGD。我们将模型的所有参数注册到优化器中。

In [4]:
# 设置一个SGD的优化器，并设置了学习率和动量
optim = torch.optim.SGD(model.parameters(), lr = 1e-2, momentum=0.9)

最后，我们调用 .step() 来启动梯度下降。优化器通过 .grad 中存储的梯度来调整每个参数。

In [5]:
optim.step()

In [6]:
a = torch.tensor([2.,3.], requires_grad=True)
b = torch.tensor([6.,4.], requires_grad=True)

In [7]:
Q = 3*a**3 - b**2

In [8]:
external_grad = torch.tensor([1.,1.])
Q.backward(gradient=external_grad)

In [9]:
print(a.grad)
print(b.grad)
print(a.grad_fn)
print(Q.grad_fn)

tensor([36., 81.])
tensor([-12.,  -8.])
None


## **计算图**
从概念上讲，autograd 会记录数据（张量）以及所有已执行的运算（以及产生的新张量），这些记录保存在一个由 Function 对象构成的 directed acyclic graph (DAG) 中。在这个 DAG 中，叶子节点是输入张量，根节点是输出张量。通过从根节点到叶子节点的追踪。该DAG图是动态的。

torch.autograd跟踪所有设置了requires_grad标志为True的张量的操作。对于不需要梯度的张量，将其设置为False就可以从梯度计算DAG中排除

In [14]:
# 解除梯度计算 -- 不计算梯度的参数称为冻结参数
x = torch.rand(5, 5)
y = torch.rand(5, 5)
z = torch.rand((5, 5), requires_grad=True)

a = x + y
print(f"Does 'a' require gradients?: {a.requires_grad}")
b = x + z
print(f"Does 'b' require gradients?: {b.requires_grad}")
x.is_leaf

Does 'a' require gradients?: False
Does 'b' require gradients?: True


True

在微调过程中，冻结模型的大部分参数，仅修改分类器层以对新标签进行预测

In [11]:
from torch import nn, optim

model = resnet18(weights=ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

In [12]:
# fc = fully connected layer
model.fc = nn.Linear(512,10)

In [13]:
# 唯一计算权重及偏置的梯度
optimizer = optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)